In [2]:
import os
import torch
import torch.nn as nn
import torch.optim as optim

from PIL import Image
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision.transforms as transforms
import torchvision.models as models

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
base_path = "/content/drive/MyDrive/Best Artworks of All Time/resized/resized"

In [5]:
device = torch.device("cpu")
print("Using device:", device)

Using device: cpu


In [6]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

In [7]:
def get_artist(filename):
    return filename.split('_')[0]

# collect artists
artists = set()

for file in os.listdir(base_path):
    if file.endswith(".jpg"):
        artists.add(get_artist(file))

artists = sorted(list(artists))   # 🔥 ensures stable labels

artist_to_idx = {artist: i for i, artist in enumerate(artists)}

# simple style + genre
style_map = {artist: i % 3 for i, artist in enumerate(artists)}
genre_map = {artist: i % 2 for i, artist in enumerate(artists)}

# build dataset
data = []

for file in os.listdir(base_path):
    if file.endswith(".jpg"):
        artist = get_artist(file)

        if artist not in artist_to_idx:
            continue

        path = os.path.join(base_path, file)

        data.append((
            path,
            artist_to_idx[artist],
            style_map[artist],
            genre_map[artist]
        ))

print("Total samples:", len(data))
print("Total artists:", len(artist_to_idx))

Total samples: 8683
Total artists: 45


In [8]:
labels = [x[1] for x in data]

print("Min label:", min(labels))
print("Max label:", max(labels))
print("Expected max:", len(artist_to_idx) - 1)

Min label: 0
Max label: 44
Expected max: 44


In [32]:
class ArtDataset(Dataset):
    def __init__(self, data, transform=None):
        self.data = data
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        path, label, _, _ = self.data[idx] # Unpack only path and artist label

        image = Image.open(path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, torch.tensor(label, dtype=torch.long)

In [33]:
dataset = ArtDataset(data, transform)

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16)

In [34]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        self.model = models.resnet18(pretrained=True)
        self.model.fc = nn.Linear(self.model.fc.in_features, num_classes)

    def forward(self, x):
        return self.model(x)

In [35]:
model = SimpleCNN(len(artist_to_idx)).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [36]:
def train(model, loader):
    model.train()
    total_loss = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

In [40]:
def evaluate(model, loader):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)

            outputs = model(images)
            _, preds = torch.max(outputs, 1)

            correct += (preds.cpu() == labels).sum().item()
            total += labels.size(0)

    print("Accuracy:", correct / total)

In [38]:
import random

random.shuffle(data)
data = data[:2000]   # 🔥 use only 2000 samples

In [39]:
for epoch in range(2):
    print(f"\nEpoch {epoch+1}")
    print("Loss:", train(model, train_loader))
    evaluate(model, val_loader)


Epoch 1
Loss: 3.155773727893829
Accuracy: 0.26

Epoch 2
Loss: 2.51293040394783
Accuracy: 0.2875


In [42]:

import torch.nn.functional as F

def find_outliers(model, loader):
    model.eval()
    count = 0

    with torch.no_grad():
        for images, _ in loader:
            outputs = model(images)

            probs = F.softmax(outputs, dim=1)
            confidence, _ = torch.max(probs, dim=1)

            for c in confidence:
                if c < 0.4:
                    print("Outlier detected:", round(c.item(), 3))
                    count += 1

            break   # just check 1 batch

    print("Total outliers in batch:", count)

In [43]:
find_outliers(model, val_loader)

Outlier detected: 0.189
Outlier detected: 0.321
Outlier detected: 0.274
Total outliers in batch: 3
